# US Treasury Yield Curve Analysis: Trump Tariff Threats Impact

This notebook analyzes the dynamics of the US Government Yield Curve around Trump tariff threat announcements using FRED (Federal Reserve Economic Data) API.

In [6]:
pip install polars requests

Note: you may need to restart the kernel to use updated packages.


In [1]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)

In [2]:
import sys
import os

# Add parent directory to path to access other folders
parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)

## FRED API Setup

This notebook loads the FRED API key from a separate file to keep it secure when sharing.

### Setup Instructions:

1. **Get FRED API Key**: Register at https://fred.stlouisfed.org/docs/api/api_key.html
2. **Create API Key File**: Create the file `../data/fred/api_key.py` (relative to this notebook)
3. **Add Your Key**: In that file, add:
   ```python
   FRED_API_KEY = 'your_actual_api_key_here'
   ```

### File Structure:
```
FIN2764/
├── functions_library/
│   └── fred.py  # Download functionality 
├── module_1/
│   └── pylab_trump_tariff_threats_assignment.ipynb  # This notebook
└── data/
    └── fred/
        └── api_key.py  # Your API key file (not shared)
```

The notebook will automatically load your API key from this secure location.

In [4]:
import sys, os
print("Working directory:", os.getcwd())
print("sys.path:", sys.path)

expected_path = os.path.join(os.path.dirname(os.getcwd()), 'data', 'fred', 'api_key.py')
print("Checking:", expected_path)
print("Exists?", os.path.exists(expected_path))

Working directory: /Users/paolomastrogiacomo/Desktop/EDHEC Financial Engineering/Advanced Fixed Income And Credit/Python/FIN2764/module_1
sys.path: ['/Users/paolomastrogiacomo/Desktop/EDHEC Financial Engineering/Advanced Fixed Income And Credit/Python/FIN2764/module_1', '/opt/anaconda3/lib/python312.zip', '/opt/anaconda3/lib/python3.12', '/opt/anaconda3/lib/python3.12/lib-dynload', '', '/opt/anaconda3/lib/python3.12/site-packages', '/opt/anaconda3/lib/python3.12/site-packages/aeosa', '/Users/paolomastrogiacomo/Desktop/EDHEC Financial Engineering/Advanced Fixed Income And Credit/Python/FIN2764']
Checking: /Users/paolomastrogiacomo/Desktop/EDHEC Financial Engineering/Advanced Fixed Income And Credit/Python/FIN2764/data/fred/api_key.py
Exists? True


In [8]:
import platform
print(platform.machine())
print(platform.python_version())

x86_64
3.12.2


In [10]:
import polars as pl
print(pl.__version__)

1.33.1


In [12]:
# FRED API Configuration
# API key is loaded from a separate file for security
try:
    from data.fred.api_key import FRED_API_KEY
    print("✅ FRED API key loaded successfully from data/fred/api_key.py")
except ImportError:
    print("⚠️  Could not load FRED API key from data/fred/api_key.py")
    print("Please ensure the file exists with the variable 'FRED_API_KEY'")
    FRED_API_KEY = None

✅ FRED API key loaded successfully from data/fred/api_key.py


## Trump Tariff Threats Timeline

Key dates for analysis based on major tariff announcements during Trump administrations:

In [15]:
# Comprehensive Trump Tariff Events Timeline (Updated through August 18, 2025)
# Categorized by Trump actions/threats vs. retaliatory responses

trump_actions = {
    # Trump First Term (2017-2021)
    '2017-01-23': 'TPP Withdrawal announced',
    '2017-04-20': 'Commerce Dept. Steel dumping investigation launched',
    '2018-01-22': 'Solar panel and washing machine tariffs (30% and 20%)',
    '2018-03-01': 'Steel and Aluminum Tariffs Announced (25% steel, 10% aluminum)',
    '2018-03-22': 'China Tariffs Announced ($60B worth of Chinese goods)',
    '2018-04-03': 'China tariff list published ($50B in goods, 25%)',
    '2018-06-15': 'China Tariffs List Published (25% on $50B of goods)',
    '2018-07-06': 'First China tariffs take effect ($34B)',
    '2018-08-23': 'Second tranche China tariffs ($16B)',
    '2018-09-17': 'China Tariffs Phase 2 Announced (10% on additional $200B)',
    '2018-09-24': 'China tariffs Phase 2 implemented ($200B at 10%)',
    '2019-05-05': 'China Tariff Escalation Tweet (25% on $200B)',
    '2019-05-10': 'China tariffs raised to 25% ($200B worth)',
    '2019-08-01': 'China Tariffs Phase 3 Announced (10% on remaining $300B)',
    '2019-08-23': 'China tariff escalation tweet (existing and new tariffs)',
    '2019-09-01': 'China tariffs on $300B begin (15% on $112B)',
    '2019-12-15': 'Planned China tariff increase postponed (trade deal progress)',
    '2020-01-15': 'Phase One China trade deal signed',
    
    # Trump Second Term (2025) - Comprehensive Timeline
    '2025-01-20': 'Inauguration - pledged immediate trade system overhaul',
    '2025-01-22': 'Executive order directing tariff review within 60 days',
    '2025-02-01': 'National emergency declared - 25% tariffs on Canada/Mexico, 10% on China (IEEPA)',
    '2025-02-10': 'Steel/aluminum tariff expansion - ended exemptions, raised aluminum to 25%',
    '2025-02-13': 'Announced reciprocal tariffs on all countries with trade barriers (April implementation)',
    '2025-02-21': 'Presidential memorandum investigating digital service taxes for retaliation',
    '2025-03-04': 'Chinese "fentanyl tariff" raised to 20% (USMCA tariffs suspended)',
    '2025-04-02': 'National emergency on trade deficit - 10% baseline tariff on all countries',
    '2025-04-05': 'Universal 10% tariff takes effect on nearly all imports',
    '2025-05-02': 'China/Hong Kong exemption removed again',
    '2025-07-02': 'Trade deal announced with Vietnam',
    '2025-07-04': 'One Big Beautiful Bill Act - eliminates de minimis exemption (2027)',
    '2025-07-07': 'Reciprocal tariffs delayed to August 1',
    '2025-07-08': 'Letters sent to 14 countries (Japan, S.Korea) threatening tariffs without deals',
    '2025-07-30': 'Cathode copper exempted from tariffs',
    '2025-08-06': 'Secondary tariff on India (25% additional) for buying Russian oil',
    '2025-08-06': 'Tariff rates finalized: Canada 35%, Mexico 30%, China 34%, EU 20%, Japan 24%',
    '2025-08-15': 'Brazil hit with 50% tariffs, India total 50%, various Asian countries 25-36%',
    '2025-08-29': 'De minimis exemption ended early (originally scheduled 2027)',
}

country_responses = {
    # China Responses (First Term)
    '2018-04-02': 'China retaliatory tariffs announced ($3B US goods)',
    '2018-04-04': 'China escalates retaliation ($50B US goods, 25%)',
    '2018-06-16': 'China announces retaliatory tariffs ($50B)',
    '2018-07-06': 'China implements retaliatory tariffs ($34B)',
    '2018-08-03': 'China tariffs on $60B US goods (5-25%)',
    '2018-08-23': 'China retaliates on $16B US goods',
    '2018-09-18': 'China announces tariffs on $60B US goods',
    '2018-09-24': 'China implements tariffs on $60B US goods',
    '2019-05-13': 'China raises tariffs on $60B US goods to 25%',
    '2019-08-05': 'China allows yuan to weaken past 7/USD',
    '2019-08-23': 'China announces tariffs on $75B US goods',
    '2019-09-01': 'China implements tariffs on $75B US goods',
    '2019-12-19': 'China announces tariff reductions (Phase One)',
    
    # EU Responses (First Term)
    '2018-03-02': 'EU threatens retaliation on steel/aluminum tariffs',
    '2018-06-22': 'EU implements retaliatory tariffs ($3.2B US goods)',
    '2018-11-02': 'EU complains to WTO about US auto tariff threats',
    
    # Canada/Mexico Responses (First Term)
    '2018-05-31': 'Canada announces retaliatory tariffs ($16.6B)',
    '2018-07-01': 'Canada implements steel/aluminum retaliation',
    '2018-06-05': 'Mexico announces retaliatory tariffs (steel, pork, bourbon)',
    
    # Other Countries (First Term)
    '2018-03-26': 'India raises tariffs on US almonds, apples',
    '2018-06-21': 'Turkey raises tariffs on US goods (rice, cars, alcohol)',
    '2018-08-10': 'Russia considers tariffs on US goods',
    
    # Second Term Responses (2025) - Major Escalation
    '2025-02-02': 'Canada, Mexico, China condemn tariffs as "unjustified"',
    '2025-03-04': 'Canada implements immediate 25% counter-tariffs on $20B US imports',
    '2025-03-04': 'China retaliates with 10-15% tariffs on US agricultural imports',
    '2025-03-05': 'Mexico announces retaliatory measures (5-20% on pork, cheese, steel)',
    '2025-03-15': 'China adds 15 US companies to export control list',
    '2025-04-04': 'EU announces $100B counter-tariffs if 30% rate implemented',
    '2025-04-06': 'China escalates to 145% tariffs on US goods (US at 145% on China)',
    '2025-04-10': 'Brazil passes reciprocity bill allowing tariff retaliation',
    '2025-05-28': 'Court of International Trade declares Trump tariff orders invalid',
    '2025-06-15': 'Canada provinces boycott US spirits amid trade war',
    '2025-07-12': 'Japan threatens "all options on table" for tariff response',
    '2025-07-20': 'EU confirms 30% counter-tariffs on $100B US goods',
    '2025-08-01': 'Malaysian government rules out retaliatory tariffs',
    '2025-08-05': 'New Zealand announces no retaliation plans',
    '2025-08-08': 'India maintains higher tariffs than US on various products (80% rice, 50% apples)',
    '2025-08-10': 'Brazil president vows reciprocation but holds back on steel/aluminum',
    '2025-08-12': 'China continues agricultural tariff escalation (chicken, wheat, corn 15%)',
    '2025-08-15': 'EU-Canada-China coordination on $330B retaliatory measures',
}

## Treasury Yield Curve Data from FRED

We'll fetch key Treasury yields across different maturities to construct the yield curve.

### Set of yield to fetch from FRED 

In [19]:
# Treasury yield series from FRED
treasury_series = {
    '3M': 'DGS3MO',    # 3-Month Treasury Constant Maturity Rate
    '6M': 'DGS6MO',    # 6-Month Treasury Constant Maturity Rate
    '1Y': 'DGS1',      # 1-Year Treasury Constant Maturity Rate
    '2Y': 'DGS2',      # 2-Year Treasury Constant Maturity Rate
    '3Y': 'DGS3',      # 3-Year Treasury Constant Maturity Rate
    '5Y': 'DGS5',      # 5-Year Treasury Constant Maturity Rate
    '7Y': 'DGS7',      # 7-Year Treasury Constant Maturity Rate
    '10Y': 'DGS10',    # 10-Year Treasury Constant Maturity Rate
    '20Y': 'DGS20',    # 20-Year Treasury Constant Maturity Rate
    '30Y': 'DGS30',    # 30-Year Treasury Constant Maturity Rate
}

### Load functions to fetch data from FRED

In [22]:
from functions_library.module_1.fred import fetch_yield_curve_data

### Fetch the Data

Now let's fetch the Treasury yield curve data:

In [25]:
# Fetch yield curve data
yield_curve_data = fetch_yield_curve_data(FRED_API_KEY=FRED_API_KEY)

if yield_curve_data is not None:
    print("\nDataFrame info:")
    print(f"Columns: {yield_curve_data.columns}")
    print(f"Data types: {yield_curve_data.dtypes}")

Fetching 3M Treasury data (DGS3MO)...
Fetching 6M Treasury data (DGS6MO)...
Fetching 1Y Treasury data (DGS1)...
Fetching 2Y Treasury data (DGS2)...
Fetching 3Y Treasury data (DGS3)...
Fetching 5Y Treasury data (DGS5)...
Fetching 7Y Treasury data (DGS7)...
Fetching 10Y Treasury data (DGS10)...
Fetching 20Y Treasury data (DGS20)...
Fetching 30Y Treasury data (DGS30)...

✅ Successfully fetched 10/10 yield series
Data range: 2015-01-02 to 2025-08-01
Data shape: (2647, 11)

DataFrame info:
Columns: ['date', '3M', '6M', '1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '20Y', '30Y']
Data types: [Date, Float64, Float64, Float64, Float64, Float64, Float64, Float64, Float64, Float64, Float64]


In [27]:
yield_curve_data

date,3M,6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2015-01-02,0.02,0.11,0.25,0.66,1.07,1.61,1.92,2.12,2.41,2.69
2015-01-05,0.03,0.1,0.26,0.68,1.06,1.57,1.85,2.04,2.32,2.6
2015-01-06,0.03,0.1,0.25,0.65,1.02,1.5,1.78,1.97,2.25,2.52
2015-01-07,0.03,0.09,0.25,0.62,1.0,1.47,1.76,1.96,2.25,2.52
2015-01-08,0.03,0.08,0.23,0.62,1.0,1.5,1.81,2.03,2.33,2.59
…,…,…,…,…,…,…,…,…,…,…
2025-07-28,4.4,4.3,4.09,3.91,3.87,3.96,4.18,4.42,4.95,4.96
2025-07-29,4.4,4.28,4.07,3.86,3.82,3.9,4.1,4.34,4.86,4.86
2025-07-30,4.41,4.31,4.12,3.94,3.89,3.96,4.15,4.38,4.9,4.89


## Event Study Analysis

We want to analyze the abnormal yield curve change around specific events by:
- For all event dates in the sample, measure the yield curve change (relative to the yield on the event date) over a window of [-30,+30] business days.
- Measure the abnormal yield change as the average yield changes over all identified event windows vs the average of all other non-event dates.

In [29]:
 import numpy as np
import pandas as pd

WINDOW = 30  # trading days each side

def build_events_table(trump_actions: dict, country_responses: dict) -> pd.DataFrame:
    rows = []
    for d, desc in trump_actions.items():
        rows.append({'date': pd.to_datetime(d), 'description': desc, 'event_type': 'trump_action'})
    for d, desc in country_responses.items():
        rows.append({'date': pd.to_datetime(d), 'description': desc, 'event_type': 'country_response'})
    events = pd.DataFrame(rows).drop_duplicates(subset=['date', 'event_type']).sort_values('date').reset_index(drop=True)
    events['term'] = np.where(events['date'] < pd.Timestamp('2021-01-01'), 'first_term', 'second_term')
    return events

def event_study(dates: pd.Series, series: pd.Series, event_dates: pd.Series, window: int = WINDOW):
    dates = pd.DatetimeIndex(dates)
    s = pd.Series(np.asarray(series), index=dates).sort_index()
    s = s[~s.index.duplicated(keep='first')]
    n = len(s)

    event_positions = []
    for ed in pd.DatetimeIndex(event_dates):
        pos = s.index.searchsorted(ed)
        if pos < n:
            event_positions.append(pos)
    event_positions = sorted(set(event_positions))

    is_near_event = np.zeros(n, dtype=bool)
    for p in event_positions:
        lo, hi = max(0, p - window), min(n, p + window + 1)
        is_near_event[lo:hi] = True
    non_event_positions = np.where(~is_near_event)[0]
    non_event_set = set(non_event_positions.tolist())

    offsets = range(-window, window + 1)
    rows = []
    values = s.values
    for offset in offsets:
        changes = [values[p+offset]-values[p] for p in event_positions if 0 <= p+offset < n]
        event_mean = np.mean(changes) if changes else np.nan

        base_changes = [values[p+offset]-values[p] for p in non_event_positions
                         if 0 <= p+offset < n and (p+offset) in non_event_set]
        baseline_mean = np.mean(base_changes) if base_changes else np.nan

        rows.append({'offset': offset, 'event_mean_change': event_mean,
                     'n_events_used': len(changes),
                     'baseline_mean_change': baseline_mean,
                     'abnormal_change': event_mean - baseline_mean})
    return pd.DataFrame(rows).set_index('offset')

def level_slope_curvature(df: pd.DataFrame) -> pd.DataFrame:
    out = df[['date']].copy()
    out['level'] = df['10Y']
    out['slope'] = df['10Y'] - df['2Y']
    out['curvature'] = 2*df['5Y'] - (df['2Y'] + df['10Y'])
    return out

print("Functions loaded.")

Functions loaded.


In [31]:
# Convert yield_curve_data from polars to pandas if needed
if hasattr(yield_curve_data, 'to_pandas'):
    ycd = yield_curve_data.to_pandas()
else:
    ycd = yield_curve_data.copy()

ycd['date'] = pd.to_datetime(ycd['date'])
ycd = ycd.sort_values('date').reset_index(drop=True)

events = build_events_table(trump_actions, country_responses)
lsc = level_slope_curvature(ycd)

print(f"Total events: {len(events)}  (trump_action: {(events.event_type=='trump_action').sum()}, "
      f"country_response: {(events.event_type=='country_response').sum()})")
events.head()

Total events: 75  (trump_action: 36, country_response: 39)


,date,description,event_type,term
0,2017-01-23,TPP Withdrawal announced,trump_action,first_term
1,2017-04-20,Commerce Dept. Steel dumping investigation lau...,trump_action,first_term
2,2018-01-22,Solar panel and washing machine tariffs (30% a...,trump_action,first_term
3,2018-03-01,Steel and Aluminum Tariffs Announced (25% stee...,trump_action,first_term
4,2018-03-02,EU threatens retaliation on steel/aluminum tar...,country_response,first_term


In [33]:
all_event_dates = events['date']

es_10y = event_study(ycd['date'], ycd['10Y'], all_event_dates)
es_level = event_study(lsc['date'], lsc['level'], all_event_dates)
es_slope = event_study(lsc['date'], lsc['slope'], all_event_dates)
es_curv  = event_study(lsc['date'], lsc['curvature'], all_event_dates)

print("Abnormal change (10Y yield) around events, selected horizons:")
print(es_10y.loc[[-30,-10,-5,-1,0,1,5,10,30], ['event_mean_change','baseline_mean_change','abnormal_change']])

es_10y.to_csv('event_study_10y.csv')
es_level.to_csv('event_study_level.csv')
es_slope.to_csv('event_study_slope.csv')
es_curv.to_csv('event_study_curvature.csv')
print("\nSaved: event_study_10y.csv, event_study_level.csv, event_study_slope.csv, event_study_curvature.csv")

Abnormal change (10Y yield) around events, selected horizons:
        event_mean_change  baseline_mean_change  abnormal_change
offset                                                          
-30              0.050000             -0.064648         0.114648
-10              0.051379             -0.021226         0.072605
-5               0.024138             -0.009860         0.033998
-1               0.003793             -0.001768         0.005561
 0               0.000000              0.000000         0.000000
 1               0.005614              0.001768         0.003846
 5               0.007857              0.009860        -0.002002
 10             -0.013818              0.021226        -0.035044
 30             -0.012549              0.064648        -0.077197

Saved: event_study_10y.csv, event_study_level.csv, event_study_slope.csv, event_study_curvature.csv
